In [ ]:
import os
import torch
import random
import torchvision
from torch import nn,Tensor
from torch.nn import functional as F
from torch.utils.data import DataLoader, Dataset
from torchvision.datasets import VOCDetection
from torch.utils.tensorboard import SummaryWriter
from torchvision.transforms import functional as TF
from torchvision.ops import nms
from matplotlib import pyplot as plt
from matplotlib.patches import Rectangle

utlis

In [2]:
#分类预测
def cls_predictor(num_inputs, num_anchors, num_classes):
    '''分类预测'''
    return nn.Conv2d(num_inputs, num_anchors * (num_classes + 1),
                     kernel_size=3, padding=1)
#生成框预测
def bbox_predictor(num_inputs, num_anchors):
    return nn.Conv2d(num_inputs, num_anchors * 4, kernel_size=3, padding=1)
#把一个batch的所有预测值拉平[b,c(预测维),h,w]->[b,h,w,c]->[b,h*w*c]
def flatten_pred(pred):
    return torch.flatten(pred.permute(0, 2, 3, 1), start_dim=1)
#把所有featuremap的预测在一维上concat起来最终输出[B，all_predict]
def concat_preds(preds):
    return torch.cat([flatten_pred(p) for p in preds], dim=1)
#返回每个像素点的框[1, H * W * boxes_per_pixel, 4]
def multibox_prior(data, sizes, ratios):
    """
    根据特征图的每个空间位置生成 anchor。

    Args:
        data: 特征图，形状 [B, C, H, W]
        sizes: anchor 的相对尺度，例如 [0.2, 0.272]
        ratios: anchor 的宽高比 w/h，例如 [1, 2, 0.5]

    Returns:
        anchors: [1, H * W * boxes_per_pixel, 4]
                 每个框格式为 [xmin, ymin, xmax, ymax]
                 坐标为归一化坐标
    """

    device = data.device
    #这里的h,w指的是featuremap的hw
    in_height, in_width = data.shape[-2:]

    sizes = torch.tensor(
        sizes,
        dtype=torch.float32,
        device=device
    )

    ratios = torch.tensor(
        ratios,
        dtype=torch.float32,
        device=device
    )

    # 生成n+m-1个anchor，第一个ratio匹配所有size，第一个size匹配剩下ratio
    boxes_per_pixel = (
        len(sizes) + len(ratios) - 1
    )

    # ------------------------------------------------
    # 1. 计算特征图每个位置对应的 anchor 中心
    # ------------------------------------------------

    offset_h = 0.5
    offset_w = 0.5

    step_h = 1.0 / in_height
    step_w = 1.0 / in_width

    center_h = (
        torch.arange(
            in_height,
            device=device
        ) + offset_h
    ) * step_h

    center_w = (
        torch.arange(
            in_width,
            device=device
        ) + offset_w
    ) * step_w

    shift_y, shift_x = torch.meshgrid(
        center_h,
        center_w,
        indexing="ij"
    )

    shift_y = shift_y.reshape(-1)
    shift_x = shift_x.reshape(-1)

    # ------------------------------------------------
    # 2. 计算一个位置上所有 anchor 的宽和高
    # ------------------------------------------------

    # 所有 size 都搭配 ratios[0]
    w = torch.cat((
        sizes * torch.sqrt(ratios[0]),

        # sizes[0] 再搭配剩下的 ratio
        sizes[0] * torch.sqrt(ratios[1:])
    ))

    h = torch.cat((
        sizes / torch.sqrt(ratios[0]),

        sizes[0] / torch.sqrt(ratios[1:])
    ))

    # 因为归一化后的 x、y 分别相对于 width、height，
    # 非正方形特征图时需要修正宽度
    w = w * in_height / in_width

    # ------------------------------------------------
    # 3. 把宽高转换为相对中心的 xyxy
    # ------------------------------------------------

    anchor_offsets = torch.stack(
        (-w, -h, w, h),
        dim=1
    ) / 2

    # [boxes_per_pixel, 4]
    # 例如：
    # [-w/2, -h/2, w/2, h/2]

    # 每个特征图位置都有同样的一组 anchor
    anchor_offsets = anchor_offsets.repeat(
        in_height * in_width,
        1
    )
    #生成的是[h*w*boxes_per_pixel,4]
    # ------------------------------------------------
    # 4. 构造每个位置的中心坐标
    # ------------------------------------------------

    centers = torch.stack(
        (
            shift_x,
            shift_y,
            shift_x,
            shift_y
        ),
        dim=1#[h*w,4]
    )
    #[h*w*boxes,4]
    centers = centers.repeat_interleave(
        boxes_per_pixel,
        dim=0
    )

    # ------------------------------------------------
    # 5. 中心位置 + anchor 相对坐标
    # ------------------------------------------------

    anchors = centers + anchor_offsets

    # D2L 返回时额外保留一个 batch-like 维度
    #返回的是[1,h*w*boxes,4]
    return anchors.unsqueeze(0)
#返回boxes1对boxes2的[n,m]iou值
def box_iou(boxes1, boxes2):
    """
    计算两组 xyxy 边界框之间的 IoU。

    Args:
        boxes1: [N, 4]
        boxes2: [M, 4]

    Returns:
        iou: [N, M]
    """
    # 交集左上角与右下角
    top_left = torch.maximum(
        boxes1[:, None, :2],
        boxes2[None, :, :2],
    )
    bottom_right = torch.minimum(
        boxes1[:, None, 2:],
        boxes2[None, :, 2:],
    )

    intersection_wh = (bottom_right - top_left).clamp(min=0)
    intersection = intersection_wh[..., 0] * intersection_wh[..., 1]

    area1 = (
        (boxes1[:, 2] - boxes1[:, 0]).clamp(min=0)
        * (boxes1[:, 3] - boxes1[:, 1]).clamp(min=0)
    )
    area2 = (
        (boxes2[:, 2] - boxes2[:, 0]).clamp(min=0)
        * (boxes2[:, 3] - boxes2[:, 1]).clamp(min=0)
    )

    union = area1[:, None] + area2[None, :] - intersection

    return intersection / union.clamp(min=1e-6)
#每个anchor分配一个真实框，没有的返回-1，最终返回[N]
# 这里输入的n是(blocks*h*w*num_anchors)
def assign_anchor_to_bbox(anchors, gt_boxes, iou_threshold=0.5):
    """
    为每个 anchor 分配一个真实框。

    匹配规则：
    1. IoU >= iou_threshold 的 anchor 匹配其 IoU 最大的真实框。
    2. 强制为每个真实框至少分配一个 anchor。

    Args:
        anchors: 归一化 anchor，[N, 4]
        gt_boxes: 归一化真实框，[M, 4]
        iou_threshold: 正样本 IoU 阈值

    Returns:
        assigned_gt_idx: [N]
            -1 表示背景，否则表示匹配到的真实框下标。
    """
    num_anchors = anchors.shape[0]
    num_gt = gt_boxes.shape[0]
    #创建一个全1的一维向量
    assigned_gt_idx = torch.full(
        (num_anchors,),
        -1,
        dtype=torch.long,
        device=anchors.device,
    )

    if num_gt == 0:
        return assigned_gt_idx

    # [N, M]
    iou = box_iou(anchors, gt_boxes)

    # 第一阶段：超过阈值的 anchor 匹配 IoU 最大的真实框
    max_iou, max_gt_idx = iou.max(dim=1)
    #返回正样本
    positive_mask = max_iou >= iou_threshold
    #赋值对应的框标号
    assigned_gt_idx[positive_mask] = max_gt_idx[positive_mask]

    # 第二阶段：保证每个真实框至少拥有一个 anchor
    # 每次从剩余 IoU 矩阵中选择最大的 anchor-GT 组合。
    iou_copy = iou.clone()

    for _ in range(num_gt):
        max_value, flat_idx = iou_copy.reshape(-1).max(dim=0)

        if max_value < 0:
            break

        anchor_idx = flat_idx // num_gt
        gt_idx = flat_idx % num_gt

        assigned_gt_idx[anchor_idx] = gt_idx

        # 此 anchor 和此真实框不再参与后续强制匹配
        iou_copy[anchor_idx, :] = -1
        iou_copy[:, gt_idx] = -1

    return assigned_gt_idx
#anchor转成[x_c,y_c,h,w]的表现形式
def box_corner_to_center(boxes):
    """
    xyxy -> cxcywh

    Args:
        boxes: [..., 4]

    Returns:
        [..., 4]
    """
    xmin, ymin, xmax, ymax = boxes.unbind(dim=-1)

    cx = (xmin + xmax) / 2
    cy = (ymin + ymax) / 2
    width = xmax - xmin
    height = ymax - ymin

    return torch.stack((cx, cy, width, height), dim=-1)
#将真实框编码为相对于 anchor 的 SSD 回归偏移(delta)。
def delta_real(anchors, assigned_boxes, eps=1e-6):
    """
    将真实框编码为相对于 anchor 的 SSD 回归偏移。

    编码公式：
        dx = 10 * (gt_cx - anchor_cx) / anchor_w
        dy = 10 * (gt_cy - anchor_cy) / anchor_h
        dw = 5 * log(gt_w / anchor_w)
        dh = 5 * log(gt_h / anchor_h)

    Args:
        anchors: [N, 4]
        assigned_boxes: [N, 4]

    Returns:
        offsets: [N, 4]
    """
    anchor_center = box_corner_to_center(anchors)
    gt_center = box_corner_to_center(assigned_boxes)
    #限制最小值为eps，防止除0
    anchor_wh = anchor_center[:, 2:].clamp(min=eps)
    gt_wh = gt_center[:, 2:].clamp(min=eps)

    offset_xy = (
        10.0
        * (gt_center[:, :2] - anchor_center[:, :2])
        / anchor_wh
    )

    offset_wh = 5.0 * torch.log(gt_wh / anchor_wh)

    return torch.cat((offset_xy, offset_wh), dim=1)
#返回对应的bbox_mask,gt_bbox,gt_labels
def multibox_target(
    anchors,
    boxes,
    labels,
    image_size=(448, 448),
    iou_threshold=0.5,
):
    """
    为一个 batch 的 anchors 生成分类和边界框训练目标。

    Args:
        anchors:
            模型生成的归一化 anchors，形状为 [1, N, 4]
            或 [N, 4]。

        boxes:
            长度为 B 的 list。
            boxes[i] 是第 i 张图片的像素坐标真实框，
            形状为 [num_objects, 4]。

        labels:
            长度为 B 的 list。
            labels[i] 是第 i 张图片的类别标签，
            形状为 [num_objects]。
            背景为 0，VOC 前景类别为 1～20。

        image_size:
            (height, width)。

        iou_threshold:
            正样本 anchor 的 IoU 阈值。

    Returns:
        bbox_labels: [B, N * 4]
        bbox_masks:  [B, N * 4]
        cls_labels:  [B, N]
    """

    if anchors.dim() == 3:
        # 模型输出为 [1, N, 4]
        anchors = anchors.squeeze(0)

    device = anchors.device
    dtype = anchors.dtype
    image_height, image_width = image_size
    num_anchors = anchors.shape[0]

    batch_bbox_labels = []
    batch_bbox_masks = []
    batch_cls_labels = []

    #对于batch中的每个样本操作
    for gt_boxes_pixel, gt_labels in zip(boxes, labels):
        gt_boxes = gt_boxes_pixel.to(
            device=device,
            dtype=dtype,
        ).clone()

        gt_labels = gt_labels.to(
            device=device,
            dtype=torch.long,
        )

        # 像素坐标转换为 [0, 1] 归一化坐标
        if gt_boxes.numel() > 0:
            gt_boxes[:, [0, 2]] /= image_width
            gt_boxes[:, [1, 3]] /= image_height
            gt_boxes = gt_boxes.clamp(min=0.0, max=1.0)

        # 每个 anchor 匹配到的 GT 下标；背景为 -1
        #维度为[N],每个框都匹配，不匹配的值是-1
        assigned_gt_idx = assign_anchor_to_bbox(
            anchors,
            gt_boxes,
            iou_threshold=iou_threshold,
        )
        #筛选出匹配到目标框的框返回一维tensor(booling)
        positive_mask = assigned_gt_idx >= 0

        # 分类目标：默认全部为背景 0
        cls_target = torch.zeros(
            num_anchors,
            dtype=torch.long,
            device=device,
        )

        # 每个 anchor 对应的真实框
        #没有匹配的框对应的值就是[0,0,0,0]
        assigned_boxes = torch.zeros(
            (num_anchors, 4),
            dtype=dtype,
            device=device,
        )

        #在正样本维度下操作
        if positive_mask.any():
            matched_gt_idx = assigned_gt_idx[positive_mask]

            # VOCDataset 中标签已经是 1～20，因此不用再加 1
            #  gt_labels是gt_boxes对应的标签，如gt_labels[0]=8
            cls_target[positive_mask] = gt_labels[matched_gt_idx]
            #筛选出正向的框
            assigned_boxes[positive_mask] = gt_boxes[matched_gt_idx]

        # 正样本四个坐标位置均为 1，背景为 0
        bbox_mask = (
            #[N,1],expaned不占内存，-1表示不动这一维
            positive_mask.unsqueeze(1)
            .expand(-1, 4)
            .to(dtype)
        )
        #输出target delta(t)
        bbox_target = delta_real(
            anchors,
            assigned_boxes,
        )

        # 清除背景 anchor 的无意义偏移
        bbox_target = bbox_target * bbox_mask

        batch_cls_labels.append(cls_target)
        batch_bbox_labels.append(bbox_target.reshape(-1))
        batch_bbox_masks.append(bbox_mask.reshape(-1))

    return (
        torch.stack(batch_bbox_labels),  # [B, N * 4]
        torch.stack(batch_bbox_masks),   # [B, N * 4]
        torch.stack(batch_cls_labels),   # [B, N]
    )
#将[x_c,y_c,h,w]转为[x1,y1,x2,y2]
def box_center_to_corner(boxes):
    """
    cxcywh -> xyxy

    Args:
        boxes: [..., 4]

    Returns:
        [..., 4]
    """
    cx, cy, width, height = boxes.unbind(dim=-1)

    xmin = cx - width / 2
    ymin = cy - height / 2
    xmax = cx + width / 2
    ymax = cy + height / 2

    return torch.stack((xmin, ymin, xmax, ymax), dim=-1)
#预测时解码到[x1,y1,x2,y2]
def decode_boxes(anchors, bbox_preds):
    """
    将 SSD 预测的偏移量解码为归一化 xyxy 边界框。

    Args:
        anchors:
           [N, 4]

        bbox_preds:
            [N, 4]

    Returns:
        decoded_boxes:
           [N, 4]
    """
    
    anchor_center = box_corner_to_center(anchors)

    # 和 bbox_preds 广播
    anchor_xy = anchor_center[:, :2]
    anchor_wh = anchor_center[:, 2:]

    # 必须与 delta_real 中的 10 和 5 对应
    pred_xy = (
        bbox_preds[..., :2] * anchor_wh / 10.0
        + anchor_xy
    )

    pred_wh = (
        torch.exp(bbox_preds[..., 2:] / 5.0)
        * anchor_wh
    )

    decoded_boxes = box_center_to_corner(
        torch.cat((pred_xy, pred_wh), dim=-1)
    )
    return decoded_boxes
#预测时图片反归一化
def denormalize_image(image):
    """
    将归一化的 [3,H,W] 图片恢复到可显示的 [H,W,3]。
    """
    mean = torch.tensor(
        [0.485, 0.456, 0.406],
        dtype=image.dtype,
        device=image.device,
    ).view(3, 1, 1)

    std = torch.tensor(
        [0.229, 0.224, 0.225],
        dtype=image.dtype,
        device=image.device,
    ).view(3, 1, 1)

    image = image * std + mean
    image = image.clamp(0, 1)

    return image.permute(1, 2, 0).cpu().numpy()

data

In [3]:
# VOC 官方的 20 个前景类别；0 保留给背景。
VOC_CLASSES = (
    "aeroplane", "bicycle", "bird", "boat", "bottle",
    "bus", "car", "cat", "chair", "cow",
    "diningtable", "dog", "horse", "motorbike", "person",
    "pottedplant", "sheep", "sofa", "train", "tvmonitor",
)
CLASS_TO_INDEX = {name: index + 1 for index, name in enumerate(VOC_CLASSES)}
#把 torchvision 的 VOC XML 标注转换为 image、boxes、labels
class VOCDataset(Dataset):
    """把 torchvision 的 VOC XML 标注转换为 image、boxes、labels。"""

    def __init__(
        self,
        root: str,
        image_set: str = "trainval",
        image_size: tuple[int, int] = (448, 448),
        train: bool = True,
        download: bool = False,
    ) -> None:
        self.dataset = VOCDetection(
            root=root,
            year="2007",
            image_set=image_set,
            download=download,
        )
        self.image_size = image_size  # (目标高度, 目标宽度)
        self.train = train

    def __len__(self) -> int:
        return len(self.dataset)
    #用来定义voc[index]应该返回什么
    def __getitem__(self, index: int) -> tuple[Tensor, Tensor, Tensor]:
        #self.dataset[index]取一张图片的数据
        image, target = self.dataset[index]
        image = image.convert("RGB")
        original_width, original_height = image.size

        annotation = target["annotation"]
        #返回'object'的值内容，否则返回[]
        objects = annotation.get("object", [])
        if isinstance(objects, dict):
            #防止只有一个目标时解析为字典
            objects = [objects]

        boxes = []
        labels = []
        for obj in objects:
            box = obj["bndbox"]

            # VOC XML 坐标从 1 开始，这里转成从 0 开始的 xyxy。
            xmin = float(box["xmin"]) - 1
            ymin = float(box["ymin"]) - 1
            xmax = float(box["xmax"]) - 1
            ymax = float(box["ymax"]) - 1
            #在boxes储存目标框数据，labels储存目标框标签
            boxes.append([xmin, ymin, xmax, ymax])
            labels.append(CLASS_TO_INDEX[obj["name"]])

        if not boxes:
            raise ValueError(f"VOC 样本 {index} 没有目标框")

        boxes_tensor = torch.tensor(boxes, dtype=torch.float32)
        labels_tensor = torch.tensor(labels, dtype=torch.long)

        # 当前模型用 torch.stack 组成 batch，所以所有图片统一为固定大小。
        target_height, target_width = self.image_size
        scale_x = target_width / original_width
        scale_y = target_height / original_height
        boxes_tensor[:, [0, 2]] *= scale_x
        boxes_tensor[:, [1, 3]] *= scale_y
        #这里的resize是进行拉伸
        image = TF.resize(image, [target_height, target_width])

        # 水平翻转图像时，边界框也必须一起翻转。
        if self.train and torch.rand(()) < 0.5:
            #进行水平翻转
            image = TF.hflip(image)
            old_xmin = boxes_tensor[:, 0].clone()
            old_xmax = boxes_tensor[:, 2].clone()
            boxes_tensor[:, 0] = target_width - old_xmax
            boxes_tensor[:, 2] = target_width - old_xmin

        image_tensor = TF.to_tensor(image)  # [0, 255] -> [0, 1]
        image_tensor = TF.normalize(
            image_tensor,
            mean=(0.485, 0.456, 0.406),
            std=(0.229, 0.224, 0.225),
        )
        #image_tensor的维度是[3, target_height, target_width]
        #boxes_tensor的维度是[num_objects, 4]
        #label_tensor的维度是[num_objects]
        return image_tensor, boxes_tensor, labels_tensor
#返回images，boxes，labels，images为tensor形式，boxes和labels为list形式
def detection_collate(batch):
    """图片尺寸一致可以 stack；每张图框数不同，所以 boxes/labels 保留 list。"""
    images, boxes, labels = zip(*batch)
    return torch.stack(images), list(boxes), list(labels)
#image的维度是[B，3, target_height, target_width]
#boxes的维度是[B,num_objects, 4]
#label的维度是[B,num_objects]
def build_voc2007_dataloader(
    root: str,
    batch_size: int = 2,
    image_size: tuple[int, int] = (448, 448),
    image_set: str = "trainval",
    train: bool = True,
    download: bool = True,
    num_workers: int = 0,
) -> DataLoader:
    dataset = VOCDataset(
        root=root,
        image_set=image_set,
        image_size=image_size,
        train=train,
        download=download,
    )
    return DataLoader(
        dataset,
        batch_size=batch_size,
        shuffle=train,
        num_workers=num_workers,
        pin_memory=torch.cuda.is_available(),
        collate_fn=detection_collate,
    )

model

In [4]:
#一个高宽减半，通道加倍层，不同的featuremap
def down_sample_blk(in_channels, out_channels):
    blk = []
    for _ in range(2):
        blk.append(nn.Conv2d(in_channels, out_channels,
                             kernel_size=3, padding=1))
        blk.append(nn.BatchNorm2d(out_channels))
        blk.append(nn.ReLU())
        in_channels = out_channels
    blk.append(nn.MaxPool2d(2))
    return nn.Sequential(*blk)
#基本的网络模型
def base_net():
    blk = []
    num_filters = [3, 16, 32, 64]
    for i in range(len(num_filters) - 1):
        blk.append(down_sample_blk(num_filters[i], num_filters[i+1]))
    return nn.Sequential(*blk)
#规定不同层数的featuremap
def get_blk(i):
    if i == 0:
        blk = base_net()
    elif i == 1:
        blk = down_sample_blk(64, 128)
    elif i == 4:
        blk = nn.AdaptiveMaxPool2d((1,1))
    else:
        blk = down_sample_blk(128, 128)
    return blk
#返回[Y, anchors, cls_preds, bbox_preds]
def blk_forward(X, blk, size, ratio, cls_predictor, bbox_predictor):
    Y = blk(X)
    anchors = multibox_prior(Y, sizes=size, ratios=ratio)
    #[1,h*w*anchors,4]
    cls_preds = cls_predictor(Y)
    bbox_preds = bbox_predictor(Y)
    return (Y, anchors, cls_preds, bbox_preds)
class TinySSD(nn.Module):
    '''return:
            anchors:[1,N_all(numblock*h*w*numanchor),4],
            cls_preds:[B,numblock*h*w*numanchor,numclass+1],
            bbox_preds:[B,N_all(numblock*h*w*numanchor,4)]
    '''
    def __init__(self, num_classes, **kwargs):
        super().__init__(**kwargs)
        self.num_classes = num_classes
        idx_to_in_channels = [64, 128, 128, 128, 128]
        for i in range(5):
            # 即赋值语句self.blk_i=get_blk(i)
            setattr(self, f'blk_{i}', get_blk(i))
            setattr(self, f'cls_{i}', cls_predictor(idx_to_in_channels[i],
                                                    num_anchors, num_classes))
            setattr(self, f'bbox_{i}', bbox_predictor(idx_to_in_channels[i],
                                                      num_anchors))

    def forward(self, X):
        anchors, cls_preds, bbox_preds = [None] * 5, [None] * 5, [None] * 5
        for i in range(5):
            # getattr(self,'blk_%d'%i)即访问self.blk_i
            X, anchors[i], cls_preds[i], bbox_preds[i] = blk_forward(
                X, getattr(self, f'blk_{i}'), sizes[i], ratios[i],
                getattr(self, f'cls_{i}'), getattr(self, f'bbox_{i}'))
        #将所有框合在一起[1,N_all(numblock*h*w*numanchor),4]
        anchors = torch.cat(anchors, dim=1)
        #将所有cls_predict合在一起[B,N_all(numblock*h*w*num_anchor*(num_class+1)]
        cls_preds = concat_preds(cls_preds)
        #cls_preds[B,numblock*h*w*numanchor,numclass+1]
        cls_preds = cls_preds.reshape(
            cls_preds.shape[0], -1, self.num_classes + 1)
        #bbox_preds[B,N_all(numblock*h*w*numanchor,4)]
        bbox_preds = concat_preds(bbox_preds)
        return anchors, cls_preds, bbox_preds

loss

In [5]:
cls_loss = nn.CrossEntropyLoss(reduction='none')
bbox_loss = nn.SmoothL1Loss(reduction='none')
#计算分类头和回归头的加和loss，返回[B,1]
def calc_loss(cls_preds, cls_labels, bbox_preds, bbox_labels, bbox_masks):
    '''这里输入的cls_preds是[b,n(numblock*h*w*numanchor),classes]
        要求的cls_label是[b,n]
        输入的bbox_preds是[b,n*4]
        要求的bbox_lables和bbox_mask是[b,n*4]
    '''
    batch_size, num_classes = cls_preds.shape[0], cls_preds.shape[2]
    #求出每张图片的loss，返回[B,1]
    cls = cls_loss(cls_preds.reshape(-1, num_classes),
                   cls_labels.reshape(-1)).reshape(batch_size, -1).mean(dim=1)
    #这里删除背景框的数据,返回[B,1]
    bbox_raw = bbox_loss(
        bbox_preds * bbox_masks,
        bbox_labels * bbox_masks).sum(dim=1)

    num_positive = (
        cls_labels > 0
    ).sum(dim=1).clamp(min=1)

    bbox = bbox_raw / num_positive
    return cls + bbox
#计算预测正确的数量
def cls_eval(cls_preds, cls_labels):
    # 由于类别预测结果放在最后一维，argmax需要指定最后一维。
    return float((cls_preds.argmax(dim=-1).type(
        cls_labels.dtype) == cls_labels).sum())
#计算预测框与真实框的差值
def bbox_eval(bbox_preds, bbox_labels, bbox_masks):
    return float((torch.abs((bbox_labels - bbox_preds) * bbox_masks)).sum())

main

In [ ]:
sizes = [[0.2, 0.272], [0.37, 0.447], [0.54, 0.619], [0.71, 0.79],
         [0.88, 0.961]]
ratios = [[1, 2, 0.5]] * 5
num_anchors = len(sizes[0]) + len(ratios[0]) - 1
train_loader = build_voc2007_dataloader(
    root="../Faster R-CNN/data",
    batch_size=8,
    image_size=(448, 448),
    image_set="trainval",
    train=True,
    download=True,
    num_workers=0,
)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

model = TinySSD(num_classes=len(VOC_CLASSES)).to(device)

optimizer = torch.optim.SGD(
    model.parameters(),
    lr=1e-3,
    momentum=0.9,
    weight_decay=5e-4,
)

best_loss = float("inf")
best_model_path = "./checkpoints/tinyssd_best.pt"
os.makedirs(
    os.path.dirname(best_model_path),
    exist_ok=True,
)

num_epochs = 50
writer = SummaryWriter('./logs','SSD')
for epoch in range(num_epochs):
    model.train()

    epoch_loss = 0.0

    for images, boxes, labels in train_loader:
        images = images.to(device)

        anchors, cls_preds, bbox_preds = model(images)

        bbox_labels, bbox_masks, cls_labels = multibox_target(
            anchors=anchors,
            boxes=boxes,
            labels=labels,
            image_size=images.shape[-2:],
            iou_threshold=0.5,
        )

        loss = calc_loss(
            cls_preds,
            cls_labels,
            bbox_preds,
            bbox_labels,
            bbox_masks,
        ).mean()

        optimizer.zero_grad()
        loss.backward()
        optimizer.step()

        epoch_loss += loss.item()

    mean_loss = epoch_loss / len(train_loader)
    if mean_loss < best_loss:
        best_loss = mean_loss

        torch.save(
            model.state_dict(),
            best_model_path
        )
    writer.add_scalar('epoch/loss',mean_loss,epoch+1)
    print(
        f"Epoch [{epoch + 1}/{num_epochs}], "
        f"loss={mean_loss:.4f}"
    )
writer.close()

KeyboardInterrupt: 

In [ ]:
test_loader = build_voc2007_dataloader(
    root="../Faster R-CNN/data",
    batch_size=1,
    image_size=(448, 448),
    image_set="test",
    train=False,      
    download=True,
    num_workers=0,
)
dataset = test_loader.dataset

random_idx = random.randrange(len(dataset))

image, boxes, labels = dataset[random_idx]
device = torch.device(
    "cuda" if torch.cuda.is_available() else "cpu"
)

model = TinySSD(
    num_classes=len(VOC_CLASSES)
).to(device)

checkpoint = torch.load(
    best_model_path,
    map_location=device,
)

model.load_state_dict(checkpoint)
model.eval()


@torch.no_grad()
def predict_one_image(
    model,
    image,
    device,
    score_threshold=0.3,
    nms_threshold=0.5,
    pre_nms_topk=100,
    max_detections=10,
):
    model.eval()

    # [3,H,W] -> [1,3,H,W]
    images = image.unsqueeze(0).to(device)

    # anchors:    [1,N,4]
    # cls_preds:  [1,N,21]
    # bbox_preds: [1,N*4]
    anchors, cls_preds, bbox_preds = model(images)

    anchors = anchors.squeeze(0)                  # [N,4]
    bbox_preds = bbox_preds[0].reshape(-1, 4)    # [N,4]

    # 将预测偏移解码为归一化 xyxy 框
    decoded_boxes = decode_boxes(
        anchors,
        bbox_preds,
    ).clamp(0, 1)

    # 分类 logits -> 概率
    class_probs = torch.softmax(
        cls_preds[0],
        dim=-1,
    )                                             # [N,21]

    # 删除背景列，只在 20 个前景类别中选择最高概率
    foreground_probs = class_probs[:, 1:]         # [N,20]

    scores, labels = foreground_probs.max(dim=1)

    # foreground_probs 第 0 列对应 VOC 类别 1
    labels = labels + 1

    # -------------------------------------------------
    # 1. Filter：过滤低置信度预测框
    # -------------------------------------------------
    keep = scores >= score_threshold

    boxes = decoded_boxes[keep]
    scores = scores[keep]
    labels = labels[keep]

    # 去掉无效框
    widths = boxes[:, 2] - boxes[:, 0]
    heights = boxes[:, 3] - boxes[:, 1]

    valid = (
        torch.isfinite(boxes).all(dim=1)
        & torch.isfinite(scores)
        & (widths > 1e-6)
        & (heights > 1e-6)
    )

    boxes = boxes[valid]
    scores = scores[valid]
    labels = labels[valid]

    if boxes.numel() == 0:
        return (
            torch.empty((0, 4)),
            torch.empty((0,), dtype=torch.long),
            torch.empty((0,)),
        )

    # -------------------------------------------------
    # 2. NMS 前先取置信度最高的一部分框
    # -------------------------------------------------
    if scores.numel() > pre_nms_topk:
        top_indices = scores.topk(pre_nms_topk).indices

        boxes = boxes[top_indices]
        scores = scores[top_indices]
        labels = labels[top_indices]

    # -------------------------------------------------
    # 3. 按类别执行 NMS
    # 不同类别之间不会互相删除
    # -------------------------------------------------
    kept_indices = []

    for class_id in labels.unique():
        class_indices = torch.where(labels == class_id)[0]

        class_keep = nms(
            boxes[class_indices],
            scores[class_indices],
            nms_threshold,
        )

        kept_indices.append(class_indices[class_keep])

    kept_indices = torch.cat(kept_indices)

    # NMS 后按照置信度从高到低排序
    order = scores[kept_indices].argsort(descending=True)
    kept_indices = kept_indices[order]

    # 限制最终框的数量
    kept_indices = kept_indices[:max_detections]

    return (
        boxes[kept_indices].cpu(),
        labels[kept_indices].cpu(),
        scores[kept_indices].cpu(),
    )
def show_prediction(
    image,
    boxes,
    labels,
    scores,
):

    display_image = denormalize_image(image)

    image_height, image_width = image.shape[-2:]

    # 归一化坐标转换为 448×448 像素坐标
    pixel_boxes = boxes.clone()

    if pixel_boxes.numel() > 0:
        pixel_boxes[:, [0, 2]] *= image_width
        pixel_boxes[:, [1, 3]] *= image_height

    fig, ax = plt.subplots(figsize=(10, 10))
    ax.imshow(display_image)

    for box, label, score in zip(pixel_boxes, labels, scores):
        x1, y1, x2, y2 = box.tolist()

        rectangle = Rectangle(
            (x1, y1),
            x2 - x1,
            y2 - y1,
            fill=False,
            edgecolor="red",
            linewidth=2,
        )
        ax.add_patch(rectangle)

        class_name = VOC_CLASSES[label.item() - 1]

        ax.text(
            x1,
            y1,
            f"{class_name}: {score.item():.2f}",
            color="white",
            fontsize=10,
            bbox={
                "facecolor": "red",
                "alpha": 0.7,
                "pad": 2,
            },
        )

    ax.set_title(
        f"{'SSD detection'} | detections={len(boxes)}"
    )
    ax.axis("off")
    plt.show()
boxes,labels,scores = predict_one_image(model=model,
                  image=image,
                  device=device,
                  score_threshold=0.3,
                  nms_threshold=0.5,
                  pre_nms_topk=100,
                  max_detections=10)
show_prediction(image=image,
                boxes=boxes,
                labels=labels,
                scores=scores)